In [1]:
# 📈 Portfolio Optimization with Modern Portfolio Theory (MPT)
# Complete implementation with US Tech Stocks

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# 💼 Step 1: Choose Your Stocks (US Tech Stocks)
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META']  # You can customize this list
# Ensure stock_names list has the same length as tickers
stock_names = ['Apple', 'Microsoft', 'Alphabet (Google)', 'Amazon', 'Meta (Facebook)']

print("Selected Stocks:")
for i, (ticker, name) in enumerate(zip(tickers, stock_names)):
    print(f"{i+1}. {name} ({ticker})")

# Define Portfolio Performance Functions - These can stay outside as they are function definitions
def portfolio_return(weights, mean_returns):
    """Calculate portfolio expected return"""
    return np.sum(weights * mean_returns)

def portfolio_volatility(weights, cov_matrix):
    """Calculate portfolio volatility (standard deviation)"""
    return np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))

def portfolio_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate=0.06):
    """Calculate portfolio Sharpe ratio"""
    port_return = portfolio_return(weights, mean_returns)
    port_volatility = portfolio_volatility(weights, cov_matrix)
    # Avoid division by zero if volatility is 0
    if port_volatility == 0:
        return 0
    return (port_return - risk_free_rate) / port_volatility

def negative_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate=0.06):
    """Negative Sharpe ratio for minimization"""
    return -portfolio_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate)

print("\n⚙️ Portfolio performance functions defined successfully!")

# 📥 Step 2: Download Historical Data
print("\n📥 Downloading historical data...")
try:
    # Define the date range
    start_date = '2018-01-01'
    end_date = '2023-01-01' # yfinance gets data up to the day before this date

    # Download the full data first
    full_data = yf.download(tickers, start=start_date, end=end_date)

    print(f"Downloaded full data shape: {full_data.shape}")
    print("Downloaded full data head:")
    print(full_data.head()) # Print head of full data
    print("Downloaded full data tail:")
    print(full_data.tail()) # Print tail of full data immediately after download

    # Check if the downloaded data contains the 'Adj Close' column level
    if isinstance(full_data.columns, pd.MultiIndex) and 'Adj Close' in full_data.columns.get_level_values(0):
        data = full_data['Adj Close']
        print("Successfully extracted 'Adj Close' data from multi-level index.")

    # Adding a fallback check just in case, though less likely for these tickers
    elif 'Adj Close' in full_data.columns:
         data = full_data['Adj Close']
         print("Successfully extracted 'Adj Close' data from single-level index.") # Should not happen with multiple tickers

    else:
         # This block is executed if 'Adj Close' column/level is not found
         # Print available columns to help diagnose
         print("\nDownloaded data structure:")
         print(full_data.columns)
         raise KeyError("Required 'Adj Close' column/level not found in the downloaded data structure. Check tickers, date range, and Yahoo Finance data availability.")


    print("Data processed successfully!")
    print(f"Processed data shape ('Adj Close'): {data.shape}")
    print("\nLast 5 days of processed data ('Adj Close'):")
    print(data.tail()) # Print tail of the 'Adj Close' data

    # Check if data is empty after download and extraction (e.g., if tickers are invalid or no data in range)
    if data.empty:
        raise ValueError("No data downloaded or extracted for the specified tickers and date range.")

    # Ensure the number of columns in 'data' matches the number of tickers *downloaded*
    # Note: yfinance might return data for fewer tickers if some fail.
    # Let's use the columns from the downloaded data as the authoritative list of *available* tickers
    available_tickers_in_data = data.columns.tolist()
    if len(available_tickers_in_data) != len(tickers):
         print(f"Warning: Number of data columns ({len(available_tickers_in_data)}) does not match initial number of tickers ({len(tickers)}).")
         print(f"Available tickers in data: {available_tickers_in_data}")
         # Update tickers and stock_names lists to only include those for which data was successfully downloaded
         original_tickers = tickers
         original_stock_names = stock_names
         tickers = available_tickers_in_data
         # Realign stock_names based on the available tickers
         stock_names = [original_stock_names[original_tickers.index(t)] for t in tickers if t in original_tickers]
         # Handle potential mismatch in names if tickers were manually added/changed
         if len(stock_names) != len(tickers):
             print("Warning: Could not perfectly realign stock names with available tickers.")
             stock_names = tickers # Fallback to using tickers as names


    # 🔁 Step 3: Calculate Daily Returns
    print("\n🔁 Calculating daily returns...")
    # Calculate returns BEFORE dropping NaNs to see the intermediate result
    returns_with_nan = data.pct_change()
    print(f"Returns shape before dropna: {returns_with_nan.shape}") # Print shape before dropna
    print("Returns head before dropna:")
    print(returns_with_nan.head()) # Print head before dropna
    print("Returns tail before dropna:")
    print(returns_with_nan.tail()) # Print tail before dropna


    returns = returns_with_nan.dropna() # Now drop NaNs


    # Check if returns dataframe is empty after dropping NaNs
    if returns.empty:
        # If returns is empty, check the original data shape to see if it was the download or the dropna
        print(f"Original data shape ('Adj Close') before calculating returns: {data.shape}")
        raise ValueError("Insufficient data to calculate returns after dropping NaNs. Check your dates, tickers, and potential data gaps.")

    print(f"Returns shape after dropna: {returns.shape}") # Print shape of returns
    print("\nDaily returns statistics:")
    print(returns.describe())

    # Visualize returns
    # Use the 'tickers' list which now contains only the available tickers
    plot_tickers = tickers
    plot_stock_names = stock_names # Use the updated stock_names list


    plt.figure(figsize=(12, 8))
    # Adjust subplot grid size based on number of available tickers
    n_plots = len(plot_tickers)
    n_cols = min(3, n_plots)
    n_rows = (n_plots + n_cols - 1) // n_cols # Calculate rows needed

    for i, ticker in enumerate(plot_tickers):
        # Ensure the ticker column exists in the returns DataFrame
        if ticker in returns.columns:
            plt.subplot(n_rows, n_cols, i+1)
            plt.hist(returns[ticker], bins=50, alpha=0.7, color=plt.cm.Set3(i % plt.cm.Set3.N)) # Use modulo for color
            plt.title(f'{plot_stock_names[i]} Returns')
            plt.xlabel('Daily Return')
            plt.ylabel('Frequency')
        else:
            print(f"Warning: Data for {ticker} not found in returns for plotting, skipping histogram.")

    plt.tight_layout()
    plt.show()


    # 📊 Step 4: Calculate Mean Returns and Covariance
    print("\n📊 Calculating mean returns and covariance matrix...")

    # Annualized mean returns (assuming 252 trading days)
    mean_returns = returns.mean() * 252
    print("\nAnnualized Mean Returns:")
    # Iterate based on mean_returns index (which are the available tickers)
    if isinstance(mean_returns, pd.Series):
         for ticker_label, ret in mean_returns.items():
             # Try to find the corresponding stock name from the updated stock_names list
             try:
                  name_index = tickers.index(ticker_label) # Use the updated tickers list
                  name = stock_names[name_index]
             except ValueError:
                  name = ticker_label # Fallback to ticker if name not found
             print(f"{name}: {ret:.2%}")
    else:
         print("Mean returns could not be calculated or is not a Series.")


    # Annualized covariance matrix
    cov_matrix = returns.cov() * 252
    print(f"\nCovariance Matrix shape: {cov_matrix.shape}")
    print("\nCovariance Matrix:")
    # print(cov_matrix) # Might be large, print only if needed

    # Correlation matrix for better understanding
    corr_matrix = returns.corr()
    plt.figure(figsize=(10, 8))
    plt.imshow(corr_matrix, cmap='coolwarm', interpolation='nearest')
    plt.colorbar()
    plt.title('Stock Correlation Matrix')
    # Use the index from corr_matrix (available tickers) for labels
    corr_labels = corr_matrix.index.tolist() # These are the available tickers
    # Attempt to map back to updated stock names for display
    corr_display_labels = [stock_names[tickers.index(t)] if t in tickers else t for t in corr_labels]

    plt.xticks(range(len(corr_labels)), corr_display_labels, rotation=45)
    plt.yticks(range(len(corr_labels)), corr_display_labels)
    # Add checks for matrix dimensions before annotating
    if corr_matrix.shape[0] == len(corr_labels) and corr_matrix.shape[1] == len(corr_labels):
        for i in range(len(corr_labels)):
            for j in range(len(corr_labels)):
                plt.text(j, i, f'{corr_matrix.iloc[i, j]:.2f}',
                        ha='center', va='center', color='white' if abs(corr_matrix.iloc[i, j]) > 0.5 else 'black')
    else:
        print("Warning: Correlation matrix dimensions do not match labels, skipping annotation.")

    plt.tight_layout()
    plt.show()

    # ⚙️ Step 5: Test with equal weights (now inside try block)
    print("\n⚙️ Testing with equal weights...")
    # Use the number of *available* assets from the returns DataFrame (which is now also the length of the updated 'tickers' list)
    num_assets_available = len(tickers) # Using the updated tickers list
    if num_assets_available > 0:
        equal_weights = np.array([1/num_assets_available] * num_assets_available)

        # Ensure mean_returns and cov_matrix are defined and have correct dimensions before use
        # Check dimensions against the number of AVAILABLE assets
        if ('mean_returns' in locals() and 'cov_matrix' in locals() and
             len(equal_weights) == num_assets_available and cov_matrix.shape[0] == cov_matrix.shape[1] == num_assets_available):

            print(f"\nEqual Weight Portfolio Performance (based on {num_assets_available} available assets):")
            equal_return_avail = portfolio_return(equal_weights, mean_returns)
            equal_volatility_avail = portfolio_volatility(equal_weights, cov_matrix)
            sharpe_equal_avail = portfolio_sharpe_ratio(equal_weights, mean_returns, cov_matrix) # Calculate once
            print(f"Expected Annual Return: {equal_return_avail:.2%}")
            print(f"Annual Volatility: {equal_volatility_avail:.2%}")
            print(f"Sharpe Ratio: {sharpe_equal_avail:.3f}")
        else:
            print("Warning: Could not calculate equal weight portfolio performance. Data inconsistencies detected.")
            # Set default values or raise an error if subsequent steps depend on these
            # For now, we'll let the later checks handle potential missing variables
            equal_weights = None # Indicate failure
            equal_return_avail, equal_volatility_avail, sharpe_equal_avail = None, None, None # Use distinct variable names

    else:
        print("Warning: No available assets after data processing. Skipping equal weight performance calculation.")
        equal_weights = None
        equal_return_avail, equal_volatility_avail, sharpe_equal_avail = None, None, None # Use distinct variable names


    # 🚧 Step 6: Optimization Constraints (now inside try block)
    print("\n🚧 Setting up optimization constraints...")
    # Use the number of *available* assets
    num_assets_available = len(tickers) # Using the updated tickers list

    if num_assets_available > 0:
        # Constraints: weights sum to 1
        constraints = ({'type': 'eq', 'fun': lambda x: np.sum(x) - 1})

        # Bounds: each weight between 0 and 1 (long-only portfolio)
        bounds = tuple((0, 1) for _ in range(num_assets_available))

        # Initial guess: equal weights based on available assets
        initial_guess = np.array([1/num_assets_available] * num_assets_available)

        print("Optimization constraints:")
        print("- Sum of weights = 1 (fully invested)")
        print("- Each weight between 0 and 1 (long-only)")
        print(f"- Initial guess: equal weights ({1/num_assets_available:.3f} each)")

        # 🚀 Step 7: Optimize the Portfolio (now inside try block)
        print("\n🚀 Optimizing portfolio for maximum Sharpe ratio...")

        # Ensure necessary variables are defined and have correct dimensions before optimization
        # Check dimensions against the number of AVAILABLE assets
        if ('mean_returns' in locals() and 'cov_matrix' in locals() and 'initial_guess' in locals() and
            'bounds' in locals() and 'constraints' in locals() and
            len(mean_returns) == num_assets_available and cov_matrix.shape[0] == cov_matrix.shape[1] == num_assets_available and
            len(initial_guess) == num_assets_available and len(bounds) == num_assets_available):

            result = minimize(
                negative_sharpe_ratio,
                initial_guess,
                args=(mean_returns, cov_matrix, 0.06),  # Using a generic 6% risk-free rate
                method='SLSQP',
                bounds=bounds,
                constraints=constraints
            )

            if result.success:
                optimal_weights = result.x
                print("Optimization successful!")

                # ✅ Step 8: Show Optimized Portfolio (now inside try block)
                print("\n✅ OPTIMIZED PORTFOLIO RESULTS")
                print("=" * 50)

                print("\nOptimal Portfolio Weights:")
                # Use the available tickers (which are now in the updated 'tickers' list) and their names
                portfolio_df = pd.DataFrame({
                    'Stock': stock_names, # Use the updated stock_names list
                    'Ticker': tickers, # Use the updated tickers list
                    'Weight': optimal_weights,
                    'Percentage': optimal_weights * 100
                })
                portfolio_df = portfolio_df.sort_values('Weight', ascending=False)
                print(portfolio_df.to_string(index=False, float_format='%.4f'))

                # Portfolio metrics
                optimal_return = portfolio_return(optimal_weights, mean_returns)
                optimal_volatility = portfolio_volatility(optimal_weights, cov_matrix)
                optimal_sharpe = portfolio_sharpe_ratio(optimal_weights, mean_returns, cov_matrix)

                print(f"\nOPTIMAL PORTFOLIO PERFORMANCE:")
                print(f"Expected Annual Return: {optimal_return:.2%}")
                print(f"Annual Volatility: {optimal_volatility:.2%}")
                print(f"Sharpe Ratio: {optimal_sharpe:.3f}")

                # Comparison with equal weight portfolio
                print(f"\nCOMPARISON WITH EQUAL WEIGHT PORTFOLIO:")
                # Ensure equal weight variables calculated based on available data are present
                if all(v is not None for v in [equal_return_avail, equal_volatility_avail, sharpe_equal_avail]):

                    comparison_df = pd.DataFrame({
                        'Metric': ['Expected Return', 'Volatility', 'Sharpe Ratio'],
                        'Equal Weight': [f"{equal_return_avail:.2%}", f"{equal_volatility_avail:.2%}", f"{sharpe_equal_avail:.3f}"],
                        'Optimized': [f"{optimal_return:.2%}", f"{optimal_volatility:.2%}", f"{optimal_sharpe:.3f}"],
                        'Improvement': [
                            f"{optimal_return - equal_return_avail:.2%}",
                            f"{optimal_volatility - equal_volatility_avail:.2%}",
                            f"{optimal_sharpe - sharpe_equal_avail:.3f}"
                        ]
                    })
                    print(comparison_df.to_string(index=False))
                else:
                     print("Warning: Could not perform comparison with equal weight portfolio due to missing or inconsistent data.")


                # Visualization (now inside try block)
                fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 12))

                # Portfolio weights pie chart
                # Use the names of the available stocks for the pie chart labels
                pie_labels = portfolio_df['Stock'].tolist()
                colors = plt.cm.Set3(np.linspace(0, 1, len(pie_labels)))
                ax1.pie(optimal_weights, labels=pie_labels, autopct='%1.1f%%', colors=colors, startangle=90)
                ax1.set_title('Optimal Portfolio Allocation')

                # Individual stock returns vs weights
                # Ensure mean_returns and cov_matrix are available and match available tickers
                if ('mean_returns' in locals() and 'cov_matrix' in locals() and
                    len(mean_returns) == len(optimal_weights) and cov_matrix.shape[0] == len(optimal_weights)):

                    stock_volatilities = np.sqrt(np.diag(cov_matrix)) # Ensure stock_volatilities is defined
                    ax2.scatter(mean_returns, optimal_weights, s=100, c=colors, alpha=0.7)
                    # Annotate using the labels from the mean_returns index (available tickers)
                    for i, ticker_label in enumerate(mean_returns.index):
                         # Use the updated stock_names list for annotation
                         try:
                             name_index = tickers.index(ticker_label)
                             name = stock_names[name_index]
                         except ValueError:
                             name = ticker_label # Fallback
                         ax2.annotate(name, (mean_returns.iloc[i], optimal_weights[i]),
                                     xytext=(5, 5), textcoords='offset points', fontsize=8)
                    ax2.set_xlabel('Expected Annual Return')
                    ax2.set_ylabel('Portfolio Weight')
                    ax2.set_title('Weight vs Expected Return')
                    ax2.grid(True, alpha=0.3)
                else:
                     print("Warning: Data for individual stock returns vs weights plot is inconsistent.")


                # Risk-Return scatter
                # Ensure mean_returns and cov_matrix are available and match available tickers
                if ('mean_returns' in locals() and 'cov_matrix' in locals() and
                    len(mean_returns) == cov_matrix.shape[0]):
                    stock_volatilities = np.sqrt(np.diag(cov_matrix))
                    ax3.scatter(stock_volatilities, mean_returns, s=100, c=colors, alpha=0.7)
                    # Annotate using the labels from the mean_returns index (available tickers)
                    for i, ticker_label in enumerate(mean_returns.index):
                        # Use the updated stock_names list for annotation
                        try:
                            name_index = tickers.index(ticker_label)
                            name = stock_names[name_index]
                        except ValueError:
                            name = ticker_label # Fallback
                        ax3.annotate(name, (stock_volatilities[i], mean_returns.iloc[i]),
                                    xytext=(5, 5), textcoords='offset points', fontsize=8)

                    ax3.scatter(optimal_volatility, optimal_return, s=200, c='red', marker='*',
                               label='Optimal Portfolio', edgecolors='black', linewidth=2)
                     # Ensure equal_volatility and equal_return are available for plotting
                    if 'equal_volatility_avail' in locals() and 'equal_return_avail' in locals() and equal_volatility_avail is not None:
                         ax3.scatter(equal_volatility_avail, equal_return_avail, s=200, c='blue', marker='s',
                                    label='Equal Weight Portfolio', edgecolors='black', linewidth=2)
                    else:
                         print("Warning: Equal weight portfolio data not available for Risk-Return scatter plot.")
                    ax3.set_xlabel('Volatility (Risk)')
                    ax3.set_ylabel('Expected Return')
                    ax3.set_title('Risk-Return Profile')
                    ax3.legend()
                    ax3.grid(True, alpha=0.3)
                else:
                     print("Warning: Data for Risk-Return scatter plot is inconsistent.")


                # Monte Carlo simulation for efficient frontier (now inside try block)
                # Use the number of available assets for simulation
                num_assets_sim = len(tickers) # Use the updated tickers list
                if num_assets_sim > 0:
                    num_portfolios = 10000
                    results = np.zeros((3, num_portfolios))

                    np.random.seed(42)
                    # Ensure mean_returns and cov_matrix are available for simulation
                    if ('mean_returns' in locals() and 'cov_matrix' in locals() and
                         len(mean_returns) == num_assets_sim and cov_matrix.shape[0] == num_assets_sim):
                         for i in range(num_portfolios):
                             weights = np.random.random(num_assets_sim)
                             weights /= np.sum(weights)

                             results[0, i] = portfolio_return(weights, mean_returns)
                             results[1, i] = portfolio_volatility(weights, cov_matrix)
                             results[2, i] = portfolio_sharpe_ratio(weights, mean_returns, cov_matrix)

                         ax4.scatter(results[1], results[0], c=results[2], cmap='viridis', alpha=0.5, s=1)
                         ax4.scatter(optimal_volatility, optimal_return, s=200, c='red', marker='*',
                                    label='Optimal Portfolio', edgecolors='black', linewidth=2)
                          # Ensure equal_volatility and equal_return are available for plotting
                         if 'equal_volatility_avail' in locals() and 'equal_return_avail' in locals() and equal_volatility_avail is not None:
                             ax4.scatter(equal_volatility_avail, equal_return_avail, s=200, c='blue', marker='s',
                                        label='Equal Weight Portfolio', edgecolors='black', linewidth=2)
                             ax4.scatter(optimal_volatility, optimal_return, s=200, c='red', marker='*',
                                        label='Optimal Portfolio', edgecolors='black', linewidth=2) # Added optimal plot again here
                         else:
                             print("Warning: Equal weight portfolio data not available for Efficient Frontier scatter plot.")

                         ax4.set_xlabel('Volatility')
                         ax4.set_ylabel('Expected Return')
                         ax4.set_title('Efficient Frontier (Monte Carlo)')
                         ax4.legend()
                         colorbar = plt.colorbar(ax4.collections[0], ax=ax4)
                         colorbar.set_label('Sharpe Ratio')
                    else:
                         print("Warning: Mean returns or covariance matrix missing or inconsistent for Monte Carlo simulation.")

                else:
                     print("Warning: No available assets for Monte Carlo simulation.")


                plt.tight_layout()
                plt.show()

                # 🧠 Conclusion (now inside try block)
                print("\n🧠 CONCLUSION")
                print("=" * 50)
                print("You successfully created an optimized portfolio using Modern Portfolio Theory!")
                print("\nKey Insights:")
                print(f"• The optimal portfolio significantly outperforms equal weighting")
                # Need to check if portfolio_df has enough rows before accessing iloc[0]
                if not portfolio_df.empty:
                     # Ensure index 0 exists before accessing
                    if 0 in portfolio_df.index:
                        print(f"• {portfolio_df.iloc[0]['Stock']} has the highest allocation ({portfolio_df.iloc[0]['Percentage']:.1f}%)")
                    else:
                         print("• Could not determine the stock with the highest allocation (portfolio_df is empty or has no index 0).")

                # Ensure optimal_sharpe, optimal_return, optimal_volatility are defined
                if 'optimal_sharpe' in locals() and 'optimal_return' in locals() and 'optimal_volatility' in locals():
                    print(f"• The portfolio achieves a Sharpe ratio of {optimal_sharpe:.3f}")
                    print(f"• Expected annual return: {optimal_return:.1%} with {optimal_volatility:.1%} volatility")
                else:
                     print("Warning: Optimal portfolio metrics could not be calculated.")


                print("\nNext Steps:")
                print("• Try different time periods or risk-free rates")
                print("• Add more stocks to the analysis")
                print("• Consider transaction costs and rebalancing frequency")
                print("• Implement minimum/maximum weight constraints")
                print("• Explore risk parity or other allocation strategies")

            else:
                print("Optimization failed!")
                print(result.message)
        else:
             print("Optimization skipped: Necessary data (mean_returns, cov_matrix, etc.) for optimization is missing or has incorrect dimensions.")
    else:
        print("Optimization skipped: No available assets after data processing.")


except Exception as e:
    # This block will print the specific exception message
    print(f"Error during data download or processing: {e}")
    print("Portfolio optimization and analysis steps were skipped due to the error.")

Selected Stocks:
1. Apple (AAPL)
2. Microsoft (MSFT)
3. Alphabet (Google) (GOOGL)
4. Amazon (AMZN)
5. Meta (Facebook) (META)

⚙️ Portfolio performance functions defined successfully!

📥 Downloading historical data...


[*********************100%***********************]  5 of 5 completed

Downloaded full data shape: (1259, 25)
Downloaded full data head:
Price           Close                                                    High  \
Ticker           AAPL       AMZN      GOOGL        META       MSFT       AAPL   
Date                                                                            
2018-01-02  40.267067  59.450500  53.188866  179.840729  78.699913  40.276419   
2018-01-03  40.260063  60.209999  54.096317  183.062424  79.066162  40.802382   
2018-01-04  40.447060  60.479500  54.306454  182.725388  79.762054  40.549914   
2018-01-05  40.907581  61.457001  55.026562  185.223465  80.750954  40.994071   
2018-01-08  40.755623  62.343498  55.220840  186.640976  80.833366  41.050156   

Price                                                    ...       Open  \
Ticker           AMZN      GOOGL        META       MSFT  ...       AAPL   
Date                                                     ...              
2018-01-02  59.500000  53.326149  179.999340  79.029547  ...

In [2]:
import yfinance as yf
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# Step 1: Define tickers
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META']

# Step 2: Download data with auto_adjust=False to include 'Adj Close'
data = yf.download(tickers, start='2018-01-01', end='2023-01-01', auto_adjust=False)
data.dropna(inplace=True)

# Step 3: Use 'Adj Close' prices
adj_close_data = data['Adj Close']

# Step 4: Calculate daily returns
daily_returns = adj_close_data.pct_change().dropna()

# Step 5: Calculate mean returns and covariance matrix (ANNUALIZED)
mean_returns = daily_returns.mean() * 252
cov_matrix = daily_returns.cov() * 252

# Step 6: Portfolio performance function
def portfolio_performance(weights, mean_returns, cov_matrix):
    returns = np.dot(weights, mean_returns)
    std = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return returns, std

# Step 7: Objective function (negative Sharpe ratio)
def negative_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate=0.01):
    p_return, p_volatility = portfolio_performance(weights, mean_returns, cov_matrix)
    if p_volatility == 0:
        return 1e10  # penalize zero volatility portfolios
    sharpe = (p_return - risk_free_rate) / p_volatility
    return -sharpe

# Step 8: Optimization setup
num_assets = len(tickers)
init_guess = num_assets * [1. / num_assets]
bounds = tuple((0, 1) for _ in range(num_assets))
constraints = [{'type': 'eq', 'fun': lambda x: np.sum(x) - 1}]

# Step 9: Run optimization
opt_result = minimize(
    negative_sharpe_ratio,
    init_guess,
    args=(mean_returns, cov_matrix),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

if not opt_result.success:
    raise ValueError("Optimization failed: " + opt_result.message)

# Step 10: Extract optimized weights and performance
opt_weights = opt_result.x
opt_return, opt_volatility = portfolio_performance(opt_weights, mean_returns, cov_matrix)
opt_sharpe = (opt_return - 0.01) / opt_volatility

# Step 11: Display results
print("Optimal Portfolio Weights:")
for ticker, weight in zip(tickers, opt_weights):
    print(f"{ticker}: {weight:.2%}")

print(f"\nExpected Annual Return: {opt_return:.2%}")
print(f"Annual Volatility: {opt_volatility:.2%}")
print(f"Sharpe Ratio: {opt_sharpe:.2f}")

# Optional: Display optimization status
print(f"\nOptimization Status: {opt_result.message}")
print(f"Success: {opt_result.success}")

[*********************100%***********************]  5 of 5 completed

Optimal Portfolio Weights:
AAPL: 49.07%
MSFT: 0.00%
GOOGL: 0.00%
AMZN: 0.00%
META: 50.93%

Expected Annual Return: 27.64%
Annual Volatility: 30.35%
Sharpe Ratio: 0.88

Optimization Status: Optimization terminated successfully
Success: True


In [3]:
import yfinance as yf
import numpy as np
import pandas as pd
from scipy.optimize import minimize

# Step 1: Define tickers
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META']

# Step 2: Download data (alternative approach - download individually)
price_data = {}
for ticker in tickers:
    ticker_data = yf.download(ticker, start='2018-01-01', end='2023-01-01')
    price_data[ticker] = ticker_data['Adj Close']

# Create DataFrame from individual downloads
data = pd.DataFrame(price_data)
data.dropna(inplace=True)

# Step 3: Calculate daily returns
daily_returns = data.pct_change().dropna()

# Step 4: Calculate mean returns and covariance matrix (ANNUALIZED)
mean_returns = daily_returns.mean() * 252
cov_matrix = daily_returns.cov() * 252

# Step 5: Portfolio performance function
def portfolio_performance(weights, mean_returns, cov_matrix):
    returns = np.dot(weights, mean_returns)
    std = np.sqrt(np.dot(weights.T, np.dot(cov_matrix, weights)))
    return returns, std

# Step 6: Objective function (negative Sharpe ratio)
def negative_sharpe_ratio(weights, mean_returns, cov_matrix, risk_free_rate=0.01):
    p_return, p_volatility = portfolio_performance(weights, mean_returns, cov_matrix)
    sharpe = (p_return - risk_free_rate) / p_volatility
    return -sharpe

# Step 7: Optimization setup
num_assets = len(tickers)
init_guess = num_assets * [1. / num_assets]
bounds = tuple((0, 1) for _ in range(num_assets))
constraints = {'type': 'eq', 'fun': lambda x: np.sum(x) - 1}

# Step 8: Run optimization
opt_result = minimize(
    negative_sharpe_ratio,
    init_guess,
    args=(mean_returns, cov_matrix),
    method='SLSQP',
    bounds=bounds,
    constraints=constraints
)

# Step 9: Extract optimized weights and performance
opt_weights = opt_result.x
opt_return, opt_volatility = portfolio_performance(opt_weights, mean_returns, cov_matrix)
opt_sharpe = (opt_return - 0.01) / opt_volatility

# Step 10: Display results
print("Optimal Portfolio Weights:")
for ticker, weight in zip(tickers, opt_weights):
    print(f"{ticker}: {weight:.2%}")

print(f"\nExpected Annual Return: {opt_return:.2%}")
print(f"Annual Volatility: {opt_volatility:.2%}")
print(f"Sharpe Ratio: {opt_sharpe:.2f}")

# Optional: Display optimization status
print(f"\nOptimization Status: {opt_result.message}")
print(f"Success: {opt_result.success}")

[*********************100%***********************]  1 of 1 completed


KeyError: 'Adj Close'